# ⚡ Activation Functions
**Course:** Deep Learning  
**Lecture:** 2 — MLP, Backpropagation, and Activation Functions  
**Part:** c — Activation Functions Showcase

---

> **Where we are:** We know how to pass data forward and errors backward. Now we look closely at the non-linearities that allow the network to learn complex patterns.

**What you'll learn:**
- Mathematical definitions of 7 major activation functions
- Their derivatives and how they affect backpropagation
- Empirical comparison of training on MNIST
- The dying ReLU problem and its solutions

**Exam relevance:** ⭐⭐⭐⭐ (You MUST know the formulas, ranges, advantages, and disadvantages of all these functions.)

In [ ]:
# ── Imports ────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import pandas as pd

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 📖 1. Mathematical Definitions
Without non-linear activation functions, a deep neural network is just a linear regressor. Let's implement all major ones.

## 🔀 Your Options: Activation Functions
At this step, you can choose between several non-linearities:

| Option | Pros | Cons | When to use |
|---|---|---|---|
| **Sigmoid** | Bounded (0,1), good for probabilities | Vanishing gradients, not zero-centered | Output layer for binary classification |
| **Tanh** | Bounded (-1,1), zero-centered | Vanishing gradients | Rarely used now, sometimes in RNNs |
| **ReLU** | Solves vanishing gradient (z>0), fast | Dying ReLU problem (z<0) | Default choice for hidden layers |
| **Leaky ReLU** | Fixes Dying ReLU | Alpha is a hyperparameter | When ReLU underperforms |
| **ELU** | Smooth near zero, robust | Computationally expensive ($e^x$) | Deep networks needing high accuracy |
| **Swish** | Smooth, non-monotonic | Computationally expensive | Very deep networks (e.g. EfficientNet) |
| **Softmax** | Outputs valid probability distribution | - | Output layer for multi-class classification |

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))
def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)  # simplification of the chain rule

def tanh_fn(z):
    return np.tanh(z)
def tanh_deriv(z):
    return 1 - np.tanh(z) ** 2

def relu(z):
    return np.maximum(0, z)
def relu_deriv(z):
    return (z > 0).astype(float)  # 1 where z>0, 0 elsewhere

def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha * z)  # small slope for z<0
def leaky_relu_deriv(z, alpha=0.01):
    return np.where(z > 0, 1.0, alpha)

def elu(z, alpha=1.0):
    return np.where(z > 0, z, alpha * (np.exp(z) - 1))  # smooth for z<0
def elu_deriv(z, alpha=1.0):
    return np.where(z > 0, 1.0, alpha * np.exp(z))

def swish(z):
    return z * sigmoid(z)  # self-gated: z * sigma(z)
def swish_deriv(z):
    s = sigmoid(z)
    return s + z * s * (1 - s)  # product rule

def softmax(z):
    e = np.exp(z - z.max())  # subtract max for numerical stability
    return e / e.sum()

# Plot all activations and derivatives
z = np.linspace(-5, 5, 400)
fns = [
    ('Sigmoid',    sigmoid,    sigmoid_deriv),
    ('Tanh',       tanh_fn,    tanh_deriv),
    ('ReLU',       relu,       relu_deriv),
    ('LeakyReLU',  leaky_relu, leaky_relu_deriv),
    ('ELU',        elu,        elu_deriv),
    ('Swish',      swish,      swish_deriv),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for name, fn, dfn in fns:
    ax1.plot(z, fn(z), label=name, lw=2)
    ax2.plot(z, dfn(z), label=name, lw=2)
ax1.set_title('Activation functions'); ax1.set_ylim(-2, 3); ax1.legend(); ax1.axhline(0, color='white', lw=0.5); ax1.axvline(0, color='white', lw=0.5)
ax2.set_title('Derivatives (used in backprop)'); ax2.set_ylim(-0.5, 1.5); ax2.legend(); ax2.axhline(0, color='white', lw=0.5); ax2.axvline(0, color='white', lw=0.5)
plt.tight_layout()
plt.show()
print('All activation functions and derivatives implemented!')

In [ ]:
# ✅ SOLUTION
def sigmoid(z):
    return 1 / (1 + np.exp(-z))
def d_sigmoid(z):
    s = sigmoid(z)
    return s * (1 - s)

def tanh(z):
    return np.tanh(z)
def d_tanh(z):
    return 1 - np.tanh(z)**2

def relu(z):
    return np.maximum(0, z)
def d_relu(z):
    return np.where(z > 0, 1.0, 0.0)

def leaky_relu(z, alpha=0.1):
    return np.where(z > 0, z, alpha * z)
def d_leaky_relu(z, alpha=0.1):
    return np.where(z > 0, 1.0, alpha)

def elu(z, alpha=1.0):
    return np.where(z > 0, z, alpha * (np.exp(z) - 1))
def d_elu(z, alpha=1.0):
    return np.where(z > 0, 1.0, elu(z, alpha) + alpha)

def swish(z):
    return z * sigmoid(z)
def d_swish(z):
    s = sigmoid(z)
    return s + z * s * (1 - s)

def softmax(z):
    # Subtract max for numerical stability
    e_z = np.exp(z - np.max(z, axis=-1, keepdims=True))
    return e_z / np.sum(e_z, axis=-1, keepdims=True)

## 🔬 2. Comparison: Shape and Derivatives
Let's visualize the activation functions and their derivatives side-by-side.

In [ ]:
z = np.linspace(-5, 5, 200)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot Activations
ax1.plot(z, sigmoid(z), label='Sigmoid')
ax1.plot(z, tanh(z), label='Tanh')
ax1.plot(z, relu(z), label='ReLU')
ax1.plot(z, leaky_relu(z, alpha=0.1), label='Leaky ReLU ($\alpha=0.1$)')
ax1.plot(z, elu(z), label='ELU')
ax1.plot(z, swish(z), label='Swish')
ax1.set_title('Activation Functions')
ax1.set_xlabel('z')
ax1.set_ylabel('f(z)')
ax1.set_ylim(-1.5, 3)
ax1.legend()
ax1.axhline(0, color='black', linewidth=0.5)
ax1.axvline(0, color='black', linewidth=0.5)

# Plot Derivatives
ax2.plot(z, d_sigmoid(z), label='Sigmoid')
ax2.plot(z, d_tanh(z), label='Tanh')
ax2.plot(z, d_relu(z), label='ReLU')
ax2.plot(z, d_leaky_relu(z, alpha=0.1), label='Leaky ReLU ($\alpha=0.1$)')
ax2.plot(z, d_elu(z), label='ELU')
ax2.plot(z, d_swish(z), label='Swish')
ax2.set_title('Derivatives (Gradient flow)')
ax2.set_xlabel('z')
ax2.set_ylabel("f'(z)")
ax2.set_ylim(-0.2, 1.5)
ax2.legend()
ax2.axhline(0, color='black', linewidth=0.5)
ax2.axvline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

# Let's also compare Leaky ReLU with different alphas
plt.figure(figsize=(6, 4))
plt.plot(z, leaky_relu(z, alpha=0.01), label='$\alpha=0.01$')
plt.plot(z, leaky_relu(z, alpha=0.1), label='$\alpha=0.1$')
plt.plot(z, leaky_relu(z, alpha=0.3), label='$\alpha=0.3$')
plt.title('Leaky ReLU Variations')
plt.legend()
plt.show()

## 🔬 3. Training Comparison on MNIST
We will build identical models (except for the activation function) and train them on MNIST to see which converges fastest.

In [ ]:
# Load and prep MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 28*28).astype('float32') / 255.0
x_test = x_test.reshape(-1, 28*28).astype('float32') / 255.0
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

def build_train_model(act_name):
    print(f"Training with {act_name}...")
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation=act_name, input_shape=(784,)),
        tf.keras.layers.Dense(64, activation=act_name),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    history = model.fit(x_train, y_train, batch_size=128, epochs=5, validation_data=(x_test, y_test), verbose=0)
    return history, model

activations = ['sigmoid', 'tanh', 'relu', 'elu', 'swish']
histories = {}

for act in activations:
    h, _ = build_train_model(act)
    histories[act] = h

print("Training complete!")

In [ ]:
# Plot Training Curves
plt.figure(figsize=(10, 6))
for act, h in histories.items():
    plt.plot(h.history['val_accuracy'], label=f"{act} (Val)", marker='o')

plt.title('Validation Accuracy over 5 Epochs on MNIST')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
# Final Accuracy Table
results = []
for act, h in histories.items():
    results.append({
        'Activation': act,
        'Final Train Acc': h.history['accuracy'][-1],
        'Final Val Acc': h.history['val_accuracy'][-1]
    })

df = pd.DataFrame(results)
display(df)

## 🎓 Exam Corner

### Question 1 (Conceptual)
Why is ReLU preferred over Sigmoid in hidden layers?

<details><summary>💡 Answer</summary>

1. **No vanishing gradient for z>0**: The derivative is 1, preserving gradients during backprop.
2. **Computational efficiency**: `max(0, z)` is much cheaper to compute than `exp()`.
3. **Sparsity**: It outputs exact 0s for negative inputs, leading to sparse representations.

</details>

### Question 2 (Practical)
A neuron with a ReLU activation outputs 0 for all training samples in your dataset. What happened and how do you fix it?

<details><summary>💡 Answer</summary>

This is the **Dying ReLU** problem. If a large gradient update causes the weights to shift such that $z < 0$ for all inputs, the ReLU derivative becomes 0. The neuron will never receive gradient updates again and is "dead".
**Fixes:** Lower the learning rate, or switch to Leaky ReLU / ELU which have non-zero gradients for $z < 0$.

</details>

### Question 3 (Conceptual)
When should you use Softmax vs Sigmoid in the output layer?

<details><summary>💡 Answer</summary>

- **Sigmoid**: Use for **binary** classification or **multi-label** classification (where classes are not mutually exclusive, e.g., an image can be both "cat" and "indoor").
- **Softmax**: Use for **multi-class** classification (where classes are mutually exclusive, e.g., MNIST digit 0-9). Softmax ensures the outputs sum to 1.0.

</details>

### Question 4 (Conceptual)
What is the gradient of ReLU at z=0?

<details><summary>💡 Answer</summary>

Mathematically, it is **undefined** (non-differentiable at exactly 0). In practice (e.g., TensorFlow/PyTorch), it is usually defined as 0 (sometimes 1). Since finding exactly `0.000000` is rare in floating point math, this doesn't cause issues.

</details>